<a href="https://colab.research.google.com/github/abdrapsandani/DataScience_240401010174_AbdullahRapsandani/blob/main/Pertemuan11_AbdullahRapsandani_240401010174.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Nama Lengkap  : Abdullah Rapsandani
#### NIM           : 240401010174
#### Kelas         : IF403

# **Sesi 11 – Unsupervised Learning Clustering**

#### STEP 1 — Generate & Eksplorasi Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
grp1 = np.random.normal([30, 20], [6, 8], (100, 2)) # hemat
grp2 = np.random.normal([70, 55], [8, 10], (100, 2)) # menengah
grp3 = np.random.normal([110, 85], [10, 8], (100, 2)) # boros
data = np.vstack([grp1, grp2, grp3])
df = pd.DataFrame(data, columns=['pendapatan_tahunan', 'skor_belanja'])
df['usia'] = np.random.randint(18, 65, len(df))
df['gender'] = np.random.choice(['L', 'P'], len(df))
print('Shape:', df.shape)
print(df.describe().round(2))
sns.scatterplot(data=df, x='pendapatan_tahunan', y='skor_belanja', alpha=0.6)
plt.title('Sebaran Pendapatan vs Skor Belanja')
plt.show()

In [ ]:
print(df["gender"].value_counts())
df["gender"].value_counts().plot(kind="bar", figsize=(5,4))
plt.title("Distribusi Gender")
plt.xlabel("Gender")
plt.ylabel("Jumlah")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(
    df[["pendapatan_tahunan","skor_belanja","usia"]].corr(),
    annot=True,
    cmap="Blues")

plt.title("Korelasi Antar Fitur")
plt.show()

#### STEP 2 — Preprocessing Data

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df[['pendapatan_tahunan', 'skor_belanja']].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Rata-rata setelah scaling:', X_scaled.mean(axis=0).round(3))
print('Std setelah scaling  :', X_scaled.std(axis=0).round(3))

#### STEP 3 — Metode Elbow untuk Menentukan K

In [ ]:
from sklearn.cluster import KMeans

wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, init='k-means++')
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.plot(range(1, 11), wcss, marker='o', color='#028090')
plt.xlabel('Jumlah Cluster (K)'); plt.ylabel('WCSS')
plt.title('Elbow Method — Segmentasi Pelanggan')
plt.show()

#### STEP 4 — Melatih Model K-Means

In [ ]:
from sklearn.metrics import silhouette_score

model = KMeans(n_clusters=3, random_state=42, init='k-means++')
model.fit(X_scaled)
df['cluster'] = model.labels_

print(f'WCSS akhir      : {model.inertia_:.3f}')
print(f'Silhouette Score: {silhouette_score(X_scaled, model.labels_):.3f}')
print(df.groupby('cluster')[['pendapatan_tahunan', 'skor_belanja']].mean().round(2))

#### STEP 5 — Visualisasi Hasil Clustering

In [ ]:
centroids = scaler.inverse_transform(model.cluster_centers_)

plt.figure(figsize=(7, 5))
plt.scatter(df['pendapatan_tahunan'], df['skor_belanja'],
            c=df['cluster'], cmap='viridis', alpha=0.7)

plt.scatter(centroids[:, 0], centroids[:, 1],
            c='red', marker='X', s=200, label='Centroid')
plt.xlabel('Pendapatan Tahunan (juta Rp)')
plt.ylabel('Skor Belanja')
plt.title('Segmentasi Pelanggan dengan K-Means')
plt.legend(); plt.show()

#### Interpretasi Tiap Cluster

*   Cluster 0 merupakan pelanggan dengan pendapatan dan tingkat belanja yang rendah sehingga dikategorikan sebagai segmen Hemat
*   Cluster 1 memiliki pendapatan dan tingkat belanja pada kategori sedang sehingga disebut segmen Menengah
*   Cluster 2 memiliki pendapatan serta tingkat belanja paling tinggi sehingga dikategorikan sebagai pelanggan Boros/Premium

#### STEP 6 — Hierarchical Clustering (Pembanding)

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

Z = linkage(X_scaled, method='ward')

plt.figure(figsize=(10,5))
dendrogram(Z)
plt.title('Dendrogram - Segmentasi Pelanggan (Ward Linkage)')
plt.xlabel('Indeks Data')
plt.ylabel('Jarak (Ward)')

plt.axhline(y=15, color='red', linestyle='--')
plt.show()

------------
# Kesimpulan

Pada praktikum ini dipelajari proses segmentasi pelanggan menggunakan metode K-Means Clustering, mulai dari eksplorasi data, preprocessing dengan StandardScaler, penentuan jumlah cluster optimal menggunakan Elbow Method, hingga evaluasi menggunakan Silhouette Score. Hasil analisis menunjukkan bahwa jumlah cluster yang sesuai adalah 3 cluster, yaitu pelanggan dengan karakteristik Hemat, Menengah, dan Boros/Premium berdasarkan pendapatan tahunan dan skor belanja.

Temuan utama menunjukkan bahwa pelanggan dapat dikelompokkan berdasarkan pola pendapatan dan perilaku belanja. Hasil K-Means juga dibandingkan dengan Hierarchical Clustering menggunakan Ward Linkage dan menunjukkan pola pengelompokan yang relatif konsisten.

Keterbatasan analisis ini adalah dataset yang digunakan merupakan data sintetis, sehingga belum tentu menggambarkan perilaku pelanggan di dunia nyata. Selain itu, clustering hanya menggunakan dua fitur utama, yaitu pendapatan tahunan dan skor belanja. Pertanyaan yang masih dapat dikembangkan adalah apakah hasil segmentasi akan tetap optimal jika ditambahkan fitur lain seperti usia, gender, frekuensi transaksi, jumlah pembelian, atau jenis produk yang dibeli.
